# Setup

In [ ]:
import os
from dataclasses import asdict

import gymnasium as gym
import numpy as np
import torch
from gymnasium.vector import AutoresetMode
from src.buffers.replay import ReplayBuffer
from src.buffers.rollout import RolloutBuffer
from src.env import HopperConfig, make_env
from src.ppo.agent import PPOAgent
from src.ppo.ppo import PPO
from src.sac.agent import SACAgent
from src.sac.sac import SAC
from src.utils import record_eval_video

n_envs = 2
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
hidden_dim = 256
seed = 42
torch.manual_seed(seed)
total_timesteps = 1_000_000
eval_interval = 8_192

video_dir = "../records/hopper"

os.makedirs(video_dir, exist_ok=True)

# Hopper

In [ ]:
env_config = HopperConfig()
envs = gym.vector.SyncVectorEnv(
    [make_env(asdict(env_config)) for _ in range(n_envs)],
    autoreset_mode=AutoresetMode.SAME_STEP,
)
eval_env = make_env(asdict(env_config))()

observation_dim = envs.single_observation_space.shape[0]
action_dim = envs.single_action_space.shape[0]

## PPO

In [ ]:
n_steps = 2048 // n_envs
mini_batch_size = 64
ppo_lr = 3e-4

ppo_agent = PPOAgent(
    observation_dim=observation_dim,
    action_dim=action_dim,
    hidden_dim=hidden_dim,
    dtype=dtype,
    device=device,
)

ppo_optimizer = torch.optim.Adam(
    ppo_agent.parameters(),
    lr=ppo_lr,
    eps=1e-5,
)

rollout_buffer = RolloutBuffer(
    buffer_size=n_steps,
    n_envs=n_envs,
    observation_dim=observation_dim,
    action_dim=action_dim,
    gamma=0.99,
    gae_lambda=0.95,
    dtype=dtype,
    device=device,
)

ppo = PPO(
    agent=ppo_agent,
    optimizer=ppo_optimizer,
    rollout_buffer=rollout_buffer,
    envs=envs,
    seed=seed,
    total_timesteps=total_timesteps,
    n_steps=n_steps,
    mini_batch_size=mini_batch_size,
    n_epochs=10,
    epsilon=0.2,
    normalize_advantage=True,
    ent_coef=0.0,
    vf_coef=0.5,
    lr=ppo_lr,
    anneal_lr=True,
    max_grad_norm=0.5,
    wandb_log=True,
    wandb_project="ppo_sac_locomotion",
    wandb_name="hopper_ppo",
    eval_interval=eval_interval,
    eval_n_episodes=10,
    dtype=dtype,
    device=device,
    eval_env=eval_env,
)

In [ ]:
train_metrics, eval_metrics = ppo.train()

In [ ]:
ppo_agent.eval()


def ppo_policy_fn(obs: np.ndarray) -> np.ndarray:
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(
        0
    )
    action = ppo_agent.predict(obs_t, deterministic=True)
    return action.squeeze(0).cpu().numpy()


returns = record_eval_video(
    config=env_config,
    policy_fn=ppo_policy_fn,
    video_dir=video_dir,
    name_prefix="ppo_final",
    num_episodes=5,
    seed=42,
)
print(f"Retorno médio: {np.mean(returns):.2f} +/- {np.std(returns):.2f}")

## SAC

In [ ]:
sac_agent = SACAgent(
    observation_dim=observation_dim,
    action_dim=action_dim,
    hidden_dim=hidden_dim,
    min_log_std=-5,
    max_log_std=2,
    dtype=dtype,
    device=device,
)

replay_buffer = ReplayBuffer(
    buffer_size=1_000_000,
    n_envs=n_envs,
    observation_dim=observation_dim,
    action_dim=action_dim,
    dtype=dtype,
    device=device,
)

sac_critic_optimizer = torch.optim.Adam(
    sac_agent.critic.parameters(),
    lr=3e-4,
    eps=1e-8,
)
sac_actor_optimizer = torch.optim.Adam(
    sac_agent.actor.parameters(),
    lr=3e-4,
    eps=1e-8,
)

sac = SAC(
    agent=sac_agent,
    critic_optimizer=sac_critic_optimizer,
    actor_optimizer=sac_actor_optimizer,
    replay_buffer=replay_buffer,
    envs=envs,
    eval_env=eval_env,
    seed=42,
    total_timesteps=total_timesteps,
    batch_size=256,
    policy_frequency=1,
    learning_starts=5_000,
    target_network_frequency=1,
    tau=0.005,
    gamma=0.99,
    alpha=0.2,
    max_grad_norm=10.0,
    eval_interval=eval_interval,
    eval_n_episodes=10,
    log_interval=1_000,
    wandb_project="ppo_sac_locomotion",
    wandb_name="hopper_sac",
    wandb_log=True,
    dtype=torch.float32,
    device="cuda",
)

In [ ]:
train_metris, eval_metrics = sac.train()

In [ ]:
sac_agent.eval()


def sac_policy_fn(obs: np.ndarray) -> np.ndarray:
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(
        0
    )
    action = sac_agent.predict(obs_t, deterministic=True)
    return action.squeeze(0).cpu().numpy()


returns = record_eval_video(
    config=env_config,
    policy_fn=sac_policy_fn,
    video_dir=video_dir,
    name_prefix="sac_final",
    num_episodes=5,
    seed=42,
)
print(f"Retorno médio: {np.mean(returns):.2f} +/- {np.std(returns):.2f}")